<a href="https://colab.research.google.com/github/Asaad972/CollabFirstNoteBook/blob/main/Tut11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required packages
!pip install -q openai anthropic langchain langchain-community langchain-openai chromadb sentence-transformers faiss-cpu numpy pandas matplotlib

import random
import time
import json
import numpy as np
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

print("\nAll libraries installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.9/405.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4

In [2]:
class ThermostatAgent:
    """A simple reflex agent that controls heating based on temperature."""
    def __init__(self, target_temp: float = 22.0):
        self.target_temp = target_temp
        self.heater_on = False

    def perceive_and_act(self, current_temp: float) -> str:
        if current_temp < self.target_temp and not self.heater_on:
            self.heater_on = True
            return f"Temperature {current_temp}°C is below target {self.target_temp}°C. Turning heater ON."
        elif current_temp >= self.target_temp and self.heater_on:
            self.heater_on = False
            return f"Temperature {current_temp}°C reached target {self.target_temp}°C. Turning heater OFF."
        else:
            status = "ON" if self.heater_on else "OFF"
            return f"Temperature {current_temp}°C. Heater status: {status}. No action needed."

# Demo
thermostat = ThermostatAgent(target_temp=22.0)
temperatures = [18.0, 19.5, 21.0, 22.0, 22.5]
for temp in temperatures:
    print(thermostat.perceive_and_act(temp))

Temperature 18.0°C is below target 22.0°C. Turning heater ON.
Temperature 19.5°C. Heater status: ON. No action needed.
Temperature 21.0°C. Heater status: ON. No action needed.
Temperature 22.0°C reached target 22.0°C. Turning heater OFF.
Temperature 22.5°C. Heater status: OFF. No action needed.


In [3]:
class RobotNavigator:
    """A model-based reflex agent that navigates while remembering obstacles."""
    def __init__(self, grid_size: int = 5):
        self.grid_size = grid_size
        self.position = [0, 0]
        self.visited = {tuple(self.position)}
        self.obstacles = set() # The "Model" of the world

    def perceive_obstacle(self, obstacle_pos: tuple):
        self.obstacles.add(obstacle_pos)
        print(f"Obstacle detected and remembered at {obstacle_pos}")

    def move(self, direction: str) -> str:
        moves = {'up': [0, 1], 'down': [0, -1], 'left': [-1, 0], 'right': [1, 0]}
        if direction not in moves: return "Invalid direction"

        new_pos = [self.position[0] + moves[direction][0], self.position[1] + moves[direction][1]]

        # Check boundaries and internal model (obstacles)
        if not (0 <= new_pos[0] < self.grid_size and 0 <= new_pos[1] < self.grid_size):
            return f"X Cannot move {direction}: out of bounds"
        if tuple(new_pos) in self.obstacles:
            return f"X Cannot move {direction}: obstacle remembered at {tuple(new_pos)}"

        self.position = new_pos
        is_new = tuple(self.position) not in self.visited
        self.visited.add(tuple(self.position))
        return f"Moved {direction} to {self.position} ({'new location' if is_new else 'visited'})"

# Demo
robot = RobotNavigator(grid_size=5)
robot.perceive_obstacle((1, 1))
for cmd in ['right', 'up', 'left', 'right']:
    print(robot.move(cmd))

Obstacle detected and remembered at (1, 1)
Moved right to [1, 0] (new location)
X Cannot move up: obstacle remembered at (1, 1)
Moved left to [0, 0] (visited)
Moved right to [1, 0] (visited)


In [4]:
class AgenticRAG:
    """An agentic RAG system with multiple specialized retrieval agents."""
    def __init__(self):
        self.agents = {}

    def register_agent(self, agent_name: str, knowledge_base: List[str]):
        self.agents[agent_name] = {'knowledge_base': knowledge_base, 'queries_handled': 0}

    def route_query(self, query: str) -> str:
        query_lower = query.lower()
        if any(word in query_lower for word in ['cloud', 'scalability']): return 'cloud_expert'
        elif any(word in query_lower for word in ['rag', 'retrieval']): return 'rag_expert'
        return 'agent_expert'

    def query(self, query: str):
        agent_name = self.route_query(query)
        agent = self.agents[agent_name]
        agent['queries_handled'] += 1

        # Simple similarity retrieval
        results = [doc for doc in agent['knowledge_base'] if any(word in doc.lower() for word in query.lower().split())]
        return {'agent': agent_name, 'query': query, 'results': results[:2]}

# Demo
rag_sys = AgenticRAG()
rag_sys.register_agent('cloud_expert', ["Cloud provides scalability for AI."])
rag_sys.register_agent('rag_expert', ["RAG grounds LLMs in facts."])

print(rag_sys.query("How does cloud help?"))

{'agent': 'cloud_expert', 'query': 'How does cloud help?', 'results': ['Cloud provides scalability for AI.']}
